# Sistema de Avaliação de Currículos usando LangGraph

## Visão Geral
Este notebook apresenta um sistema automatizado de avaliação de currículos implementado usando LangGraph e um modelo LLM. O sistema avalia currículos com base em quatro critérios principais: experiências, idioma, conquistas e habilidades.

## Motivação
Sistemas automatizados de avaliação de currículos podem agilizar significativamente o processo de avaliação em ambientes corporativos, fornecendo avaliações consistentes e objetivas. Esta implementação visa demonstrar como modelos de linguagem grandes e fluxos de trabalho baseados em grafos podem ser combinados para criar um sistema sofisticado de avaliação.

## Componentes Principais
1. Grafo de Estado: Define o fluxo de trabalho do processo de avaliação
2. Modelo LLM: Fornece a compreensão e análise de linguagem subjacente
3. Funções de Avaliação: Funções separadas para cada critério de avaliação
4. Lógica Condicional: Determina o fluxo do processo de avaliação com base em pontuações intermediárias

## Método
O sistema segue uma abordagem passo a passo para avaliar currículos:

1. Experiências Profissionais e Acadêmicas: Avalia quão bem o currículo descreve as experiências do candidato
2. Idioma: Avalia como esta descrito o conhecimento do idioma e seu nível correspondente
3. Conquistas: Verifica como está informado os cursos, trabalho voluntários entre outros reconhecimentos
4. Habilidades: Identifica até 30 habilidades desenvolvidas ou adquiridas ao longo da carreira

Cada etapa é executada condicionalmente com base nas pontuações das etapas anteriores, permitindo o término antecipado de currículos de baixa qualidade. A pontuação final é uma média ponderada de todas as pontuações dos componentes individuais.

## Conclusão
Este notebook demonstra uma abordagem flexível e extensível para a avaliação automatizada de currículos. Ao aproveitar o poder dos grandes modelos de linguagem e um fluxo de trabalho baseado em grafos, oferece uma avaliação baseada nos critérios utilizados e revisados pela Gupy, em seu manual divulgado por Gabriel Pontes.


## Instalações e configurações

## Essa célula importa as bibliotecas necessárias e configura a chave da API da OpenRouter.

In [38]:
%pip install langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import re


# Carrega as variáveis de ambiente e configura a chave da API da OpenAI
load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv('OPENROUTER_API_KEY')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Definição do estado

Esta célula define a classe State, que representa o estado do processo de avaliação.

In [46]:
class State(TypedDict):
    """ Representa o estado do processo de avaliação do currículo """
    curriculum: str
    experience_score: float
    language_score: float
    achievements_score: float
    habilities_score: float
    final_score: float

## Inicialização do modelo de linguagem

Essa célula inicializa o modelo ChatOpenAI.

In [47]:
# Inicializa o modelo ChatAnthropic
llm = ChatOpenAI(
    model="anthropic/claude-3.7-sonnet",
    openai_api_key=os.getenv('OPENROUTER_API_KEY'),
    openai_api_base="https://openrouter.ai/api/v1",
)

## Grading Functions

This cell defines the functions used in the grading process, including score extraction and individual grading components.

In [ ]:
def extract_score(content: str) -> float:
    """Extrai a pontuação numérica da resposta do LLM."""
    match = re.search(r'Pontuação:\s*(\d+(\.\d+)?)', content)
    if match:
        return float(match.group(1))
    raise ValueError(f"Não foi possível extrair a pontuação de: {content}")

def check_experience(state: State) -> State:
    """Verifica a descrição das experiências do candidato no currículo."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a descrição do currículo, validando se possuí os campos Experiência Profissional e Acadêmica."
        "Em experiência acadêmica verifique se contem o nome do curso, status, o nome da instituição e o período (MM/AAAA - MM/AAAA)."
        "Em experiência profissional, verifique o nome do cargo, onde ele precisará ter as nomenclaturas padrões de mercado, facilitando sua identificação por agentes de IA."
        "Verifique as principais responsabilidades e conquistas, enfatizando realizações concretas por meio de projetos ou resultados mensuráveis."
        "Forneça uma pontuação de idioma entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nExperiências Profissionais e Acadêmicas: {curriculum}"
    )
    result = llm.invoke(prompt.format(curriculum=state["curriculum"]))
    try:
        state["experience_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_experience: {e}")
        state["experience_score"] = 0.0
    return state

def check_language(state: State) -> State:
    """Verifica a descrição do nível de conhecimento do idioma."""
    prompt = ChatPromptTemplate.from_template(
        "Analise o idioma indicado e o correspondente nível de conhecimento no currículo."
        "Forneça uma pontuação de idioma entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nIdioma: {curriculum}"
    )
    result = llm.invoke(prompt.format(curriculum=state["curriculum"]))
    try:
        state["language_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_language: {e}")
        state["language_score"] = 0.0
    return state

def analyze_achievements(state: State) -> State:
    """Analisa a descrição das conquistas e certificados."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a descrição de cursos, trabalho voluntários, certificados e demais reconhecimentos. Ex.: softwares, metodologias, certificações, além de competências como liderança, trabalho em equipe, comunicação e etc."
        "Forneça uma pontuação dessa descrição entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nConquistas e Certificados: {curriculum}"
    )
    result = llm.invoke(prompt.format(curriculum=state["curriculum"]))
    try:
        state["achievements_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em analyze_achievements: {e}")
        state["achievements_score"] = 0.0
    return state

def analyze_habilities(state: State) -> State:
    """Avalia a descrição de habilidades técnicas e comportamentais no currículo."""
    prompt = ChatPromptTemplate.from_template(
        "Avalie se foram descritas pelo menos 30 habilidades desenvolvidas durante a trajetória profissional. "
        "Forneça uma pontuação de profundidade entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nHabilidades: {curriculum}"
    )
    result = llm.invoke(prompt.format(curriculum=state["curriculum"]))
    try:
        state["habilities_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em analyze_habilities: {e}")
        state["habilities_score"] = 0.0
    return state

def calculate_final_score(state: State) -> State:
    """Calcula a pontuação final com base nas pontuações dos componentes individuais."""
    state["final_score"] = (
        state["experience_score"] * 0.3 +
        state["language_score"] * 0.2 +
        state["achievements_score"] * 0.3 +
        state["habilities_score"] * 0.2
    )
    return state

## Definição do fluxo de trabalho

Essa célula define o fluxo de trabalho de avaliação usando StateGraph.

In [49]:
# Inicializa o StateGraph
workflow = StateGraph(State)

# Adiciona nós ao grafo
workflow.add_node("check_experience", check_experience)
workflow.add_node("check_language", check_language)
workflow.add_node("analyze_achievements", analyze_achievements)
workflow.add_node("analyze_habilities", analyze_habilities)
workflow.add_node("calculate_final_score", calculate_final_score)

# Define e adiciona arestas condicionais
workflow.add_conditional_edges(
    "check_experience",
    lambda x: "check_language" if x["experience_score"] > 0.5 else "calculate_final_score"
)
workflow.add_conditional_edges(
    "check_language",
    lambda x: "analyze_achievements" if x["language_score"] > 0.6 else "calculate_final_score"
)
workflow.add_conditional_edges(
    "analyze_achievements",
    lambda x: "analyze_habilities" if x["achievements_score"] > 0.7 else "calculate_final_score"
)
workflow.add_conditional_edges(
    "analyze_habilities",
    lambda x: "calculate_final_score"
)

# Define o ponto de entrada
workflow.set_entry_point("check_experience")

# Define o ponto de saída
workflow.add_edge("calculate_final_score", END)


# Compila o grafo
app = workflow.compile()

## Função de avaliação de currículo

Essa célula define a função principal para avaliar um currículo usando o fluxo de trabalho definido.

In [50]:
def grade_curriculum(curriculum: str) -> dict:
    """Avalia o currículo fornecido usando o fluxo de trabalho definido."""
    initial_state = State(
        curriculum=curriculum,
        experience_score=0.0,
        language_score=0.0,
        achievements_score=0.0,
        habilities_score=0.0,
        final_score=0.0
    )
    result = app.invoke(initial_state)
    return result

## Currículo de exemplo

Esse é um currículo de exemplo para testar o sistema de avaliação.

In [ ]:
sample_curriculum = """
Rafael de Novaes

Senior AI Product Manager (PO) | LLMs, RAG, LangGraph | SQL/Python | PSPO I

São Paulo - SP

Telefone: (11) 9 4910-5033 

rafaeldenovaes@gmail.com

https://www.linkedin.com/in/rafaeldenovaes/

RESUMO

Profissional de Produto Sênior que transforma IA em resultado em ambientes enterprise/B2B;
exposição a ERP (SAP – orçamento B3). Especialista em productization de LLMs (RAG, LangGraph) e
experimentação com medição de impacto; base sólida em Produto/Agile (Smiles, Deloitte, Itaú).
SQL/Python para POCs e análises. Criei o DestravaCV (OpenAI, análise multi‑vaga, 95+ testes) e o
FacilIAuto (B2B multi‑tenant, mobile‑first, setup ~30 min, ROI ~380%), com roadmap para integrações
ERP/CRM. Desenho experiências “in‑line” de baixo atrito; formação contínua: Scoras Academy
(Engenharia de IA, 2025) e GoPractice AI/ML Simulator for PMs (em andamento).

SKILLS

AI Productization: LLMs, RAG, LangGraph, OpenAI API, PydanticAI, Model Context Protocol (AWS)
Experimentação & Métricas: telemetria, definição/avaliação de impacto em métricas de negócio
Dados & POCs: SQL, Python (prototipação e análises)
Arquitetura SaaS: multi‑tenant, REST APIs, Docker, PostgreSQL
Web/Backend Stack: FastAPI (Python 3.10+), React 18, TypeScript, Chakra UI, Vite
Estratégia de Produto: priorização por ROI/adoção, Roadmap, Go‑to‑Market/Pricing
Discovery & PRD: hipóteses, user stories, critérios de aceite, documentação funcional
Product Analytics & Storytelling: KPIs, dashboards, insights acionáveis
Agilidade & Qualidade: Scrum, Kanban, OKRs, XP/TDD, testes automatizados (80%+)
Enterprise/B2B & ERP: ambientes enterprise; exposição a ERP (SAP – orçamento B3); integrações
ERP/CRM (roadmap)

IDIOMA
Inglês avançado

FORMAÇÃO
FATEC Zona Leste — Tecnólogo em Informática para a Gestão de Negócios (2009)
Scoras Academy — Engenharia de IA (2025, em andamento): LLMs; RAG; LangGraph/LLM Routing;
PydanticAI; MCP (AWS)
Pendo.io — AI for Product Management; Product Analytics Certification; (12/2024)
GoPractice — Generative AI for Product Managers – Mini Simulator (11/2024); AI/ML Simulator for
Product Managers (2025, em andamento)
 
EXPERIÊNCIA PROFISSIONAL

Consultoria Bethesda
Fundador
dez 2024 - atualmente
FacilIAuto: concepção e desenvolvimento de B2B SaaS multi‑tenant (Python 3.10+/FastAPI; React
18/TS; XP/TDD 80%+), com motor de recomendação por scoring multidimensional, feedback
iterativo (6 perfis), mobile‑first; setup ~30 min; ROI ~380%; telemetria e integrações ERP/CRM no
roadmap.
DestravaCV: plataforma de RH com IA para análise semântica “multi‑vaga” (até 7) e
compatibilidade ATS; arquitetura cloud e estratégia de monetização; pipeline CI/CD; PRD e
backlog; 95+ testes; stack Node.js, PostgreSQL, PWA, Docker, OpenAI API.
Governança de produto: discovery por hipóteses, priorização por impacto/ROI, definição de KPIs e
instrumentação para experimentação/medição de impacto.
Entregas: documentação funcional, integrações, quality gates e automação de testes para garantir
velocidade com qualidade.

Smiles
Product Owner Sênior
abr 2023 - nov 2023
Gerenciamento de projeto para desenvolvimento de um produto em uma equipe composta por 18
profissionais: Responsável pelo alinhamento das necessidades dos stakeholders, garantindo que
todos os recursos, melhorias, correções de bugs e tarefas técnicas estejam priorizados
adequadamente no desenvolvimento do produto; 
Escrita de histórias de usuários e definição dos critérios de aceite que delineiam as necessidades e
condições da área de negócios; 
Concepção e evolução de produtos digitais, como a implementação dos parques Universal Studios
no site da Smiles Viagens, nas versões navegador e mobile; 
Desenvolvimento de produtos em equipes ágeis, com uso das metodologias Scrum e Kanban;
Apoio à equipe de UX na elaboração de protótipos; 
Interface entre as áreas de Desenvolvimento e Negócios, facilitando o entendimento e as
expectativas do desenvolvimento do produto;

Luv2Mob
Product Owner / Agile Master
fev 2022 - ago 2022
Responsável pela condução da squad de cadastro/onboarding do banco digital AL5; 
Gerenciamento de backlog de produto alinhado às necessidades dos stakeholders; 
Condução de cerimônias ágeis como reuniões diárias, planejamento, sprint review e
retrospectivas; 
Responsável pela concepção e evolução de produtos digitais, como o desenvolvimento do Internet
Banking do Banco AL5, nas versões navegador e mobile; 
Planejamento de sprints, com avaliação do conjunto de tarefas e objetivos a serem atingidos em 2
semanas, com acompanhamento de burn down para avaliação de performance da squad;
Interface entre as áreas de Desenvolvimento e Negócios, facilitando o entendimento e as
expectativas do desenvolvimento do produto; 
Implementação e lançamento de funcionalidades de produto de ponta a ponta, desde a
especificação, testes de aceitação, até o acompanhamento do lançamento em produção;
Planejamento de lançamento conforme metas e métricas de produto estabelecidas em conjunto
com os stakeholders; 
Desenvolvimento de produtos em equipes ágeis, utilizando as metodologias Scrum e Kanban;
Apoio à equipe de UX na elaboração de protótipos;
Atuação em equipe composta por 19 profissionais (1 gerente comercial, 1 gerente de auditoria, 1
gerente de crédito, 1 coordenador de sistemas, 2 analistas de sistemas, 4 PO´s, 1 tech-lead e 8
desenvolvedores).

Deloitte
Gerente de Projetos
dez 2020 - fev 2022
Acompanhamento de OKRs definidos pelos stakeholders e reporte do desenvolvimento em times ágeis.
Gestão de situações conflituosas e urgentes.
Roadmap revisado trimestralmente com stakeholders, garantindo integração entre times.
Modelo de status report para projetos ágeis e cascata; referência na equipe ágil B3.
Apoio em business cases com definição, análise e acompanhamento de métricas de viabilidade e
valor.
Gestão de portfólio (seleção, priorização e controle) alinhada à squad de Garantias e à
Superintendência de Riscos Sistêmicos.
Ferramentas: Miro, Jira, Confluence, Clarity, SAP.
Equipe: 13 profissionais — 1 diretor de negócios, 2 superintendentes de negócios, 2
superintendentes de sistemas, 2 gerentes de negócios, 2 gerentes de sistemas, 1 analista de riscos
sênior, 1 analista de garantias sênior e 2 coordenadores de sistemas.

Itaú Unibanco
Analista de Projetos e Processos | Inovação e Transformação Digital
Analista de Projetos Pleno
Analista de Projetos Júnior - Business Intelligence
out 2008 - out 2020

Analista de Projetos e Processos | Inovação e Transformação Digital
Responsável pela condução de ciclos de viabilidade técnica para mapeamento de problemas,
negociação e construção de soluções para inovação e diferenciação, conforme as necessidades
dos stakeholders; Disseminação do mindset digital, visando integrar a centralidade no cliente, na cultura Lean, novas
tecnologias e práticas ágeis;
Mapeamento e modelagem de processos com uso da ferramenta Bizagi;
Elaboração de relatórios de acompanhamento da equipe com uso da ferramenta Power BI;
Responsável pela condução de dinâmicas de product discovery em todos os processos de crédito
PF e PJ, com o objetivo de abordar todos os valores do Design Thinking de forma tempestiva,
permitindo diagnosticar, reestruturar, analisar viabilidade técnica e esclarecer todas as
informações que permeiam a cadeia de valor, em alinhamento às necessidades do cliente;
Atuação em equipe composta por 12 profissionais (1 superintendente de crédito, 1 gerente de
crédito, 1 coordenador de crédito, 1 analista de crédito, 1 superintendente de cartões, 1 gerente
de cartões, 2 analistas de cartões e 3 analistas de política de cartões).

Com realizações a destacar:
Projeto de design sprint para cartões pessoa física, voltado para reestruturação e esclarecimento
de informações utilizadas para aplicação de regras, filtros e limites para os clientes,
parametrizando grupos específicos para utilização de modelos e ponderadores de cálculos,
permitindo a aplicação das políticas de forma mais transparente e segura, resultando em
aumento de 50% na eficiência do cálculo de limites pré-aprovados de cartões, no ano de 2019.

Analista de Projetos Pleno
Responsável pela implementação e lançamento de funcionalidades de projeto de ponta a ponta,
desde a especificação até o acompanhamento da entrada em produção, para a área de crédito
pessoa jurídica, auxiliando na implantação de produtos financeiros, conforme as necessidades
dos stakeholders;
Elaboração de plano de projeto e cronograma, atuando nos planos de mitigação de riscos,
levantamento de escopo, especificação, homologação, piloto e acompanhamento da entrada em
produção;
Acompanhamento de pós-implantação de projetos, com reporte sobre o andamento para a área
de crédito pessoa jurídica (varejo e atacado), utilizando as metodologias Scrum, Lean e o PMBOK;
Atuação em equipe composta por 20 profissionais (2 superintendentes de crédito, 4 gerentes de
rédito, 4 coordenadores de crédito, 2 gerentes de sistemas, 2 coordenadores de sistemas e 6
analistas de sistemas).

Com realizações a destacar:
Reestruturação da árvore de limites de crédito (usabilidade de limites), voltado para simplificar a
atuação da área comercial e das mesas de crédito na atuação das propostas de negócio para
clientes PJ, no ano de 2017;
Projeto que permitiu ao cliente PJ informar o faturamento sem a anuência do contador, através do
Bankline, no ano de 2016;
Analista de Projetos Júnior
Descrição, análise e acompanhamento de métricas e indicadores (KPIs) desenvolvidas na
ferramenta SAS;
Levantamento de requisitos, especificação, homologação, tratativa de impeditivos e
acompanhamento de pós-implantação de projetos voltados para a criação de ferramentas
gerenciais, como dashboards, datamart e base de dados SAS
    """

## Avaliação da currículo de exemplo

Essa célula demonstra como usar o sistema de avaliação no curríulo de exemplo e exibir os resultados.

In [52]:
# Avalia a redação de exemplo
result = grade_curriculum(sample_curriculum)

# Converte as pontuações de 0-1 para 0-10
final_score = result['final_score'] * 10
experience_score = result['experience_score'] * 10
language_score = result['language_score'] * 10
achievements_score = result['achievements_score'] * 10
habilities_score = result['habilities_score'] * 10

# Exibe os resultados
print(f"Pontuação Final do Currículo: {final_score:.2f}/10\n")
print(f"Pontuação de Experiências: {experience_score:.2f}/10")
print(f"Pontuação de Idioma: {language_score:.2f}/10")
print(f"Pontuação de Conquistas: {achievements_score:.2f}/10")
print(f"Pontuação de Habilidades: {habilities_score:.2f}/10")

Pontuação Final do Currículo: 8.34/10

Pontuação de Experiências: 9.00/10
Pontuação de Idioma: 7.00/10
Pontuação de Conquistas: 9.20/10
Pontuação de Habilidades: 8.00/10
